# G1：P0・P1・P2の高速歩行限界・転倒耐性テスト（Isaac Gym）

**目的：各方策が転倒せずにどこまで速く歩けるかを比較します．** 強化学習は行わず，保存済み`model_6700.pt`の推論のみを実行します．

## 実験設計

- P0（接地切替え許容幅0 ms），P1（±20 ms），P2（±40 ms）．
- 指令速度 **1.8, 1.9 m/s** （0.1 m/s刻み）．最初の粗い探索です．必要ならその後0.05 m/s刻みで再検査します．
- **各速度8試行，18秒間**．初期関節角度は基準姿勢±0.02 rad，初期の身体速度・関節速度は0，同じ初期条件で3方策を比較．
- **同じ0.8秒周期の位相入力**を与えます．接地報酬は評価時には使用せず，各方策の学習結果のみ異なります．
- 平坦床，固定摩擦・質量，観測ノイズなし，外乱なし，headless／T4 GPU推奨．
- **転倒しないことと，高速で歩けることは別です．** 非転倒率と，実際の進行方向速度が指令に追従した率の両方を報告します．

### 判定の注意

- エピソード終了（`dones`）を転倒・失敗の代理指標とします．18秒の評価は通常のエピソード上限より短く設定します．
- **走行可能速度の下限・上限を確定する試験ではありません．** 今回の初期条件と18秒，8試行での「観測された最高速度」です．
- **1.0 m/sを超える指令は学習で使った前進速度の上限を超える外挿（OOD）**です．
- `world_x`方向（初期進行方向）の速度を評価し，旋回・横歩きで「高速と見なす」誤解を避けます．
- 条件を変更したP1・P2も含め，**学習済みファイルは読み取り専用**です．

## 1．Google Drive・出力先・GPU

In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess, os, json, shutil, sys, time
from datetime import datetime

drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/Colab Notebooks/PhysicalAI応用編１/G1_RL_Training')
assert DRIVE.is_dir(), DRIVE
EXP = DRIVE / 'contact_tolerance_P0_P1_P2_from6500_20261024_075342'
assert EXP.is_dir(), f'先日のP0/P1/P2学習フォルダーが見つかりません: {EXP}'
OUT_DIR = EXP / 'evaluation' / 'high_speed_limit_P012_coarse_20261024'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('既存のP0/P1/P2結果は変更しません．高速歩行限界テストを別フォルダーに保存します．')
print('新規出力:', OUT_DIR)
subprocess.run(['nvidia-smi'],check=True)

## 2．Isaac GymとPython 3.8を準備（従来のノートブックと同じ）

In [ ]:
WORK = Path('/content')
repo = WORK / 'unitree_rl_gym'
rsl = WORK / 'rsl_rl'
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/unitreerobotics/unitree_rl_gym.git',str(repo)],check=True)
if not rsl.exists():
    subprocess.run(['git','clone','https://github.com/leggedrobotics/rsl_rl.git',str(rsl)],check=True)
subprocess.run(['git','-C',str(repo),'checkout','276801e46c5d433564f24658bac64f254b7d2d4b'],check=True)
subprocess.run(['git','-C',str(rsl),'checkout','2ad79cf0caa85b91721abfe358105f869a784121'],check=True)
print('unitree_rl_gym',subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip())
print('rsl_rl',subprocess.check_output(['git','-C',str(rsl),'rev-parse','HEAD'],text=True).strip())

In [ ]:
ARCHIVE = DRIVE / 'IsaacGym_Preview_4_Package.tar'
print('期待するファイル:', ARCHIVE)
if not ARCHIVE.is_file():
    print('ファイル未配置です．公式配布サイトから入手したtarファイルをGoogle Driveへアップロードしてください．')
else:
    print('検出:', ARCHIVE.name, '容量 GiB:', round(ARCHIVE.stat().st_size / 1024**3, 2))

In [ ]:
from pathlib import Path
import subprocess, os

MAMBA_BIN = Path('/content/bin/micromamba')
if not MAMBA_BIN.exists():
    subprocess.run(['bash','-lc',
      'mkdir -p /content && curl -fsSL https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content bin/micromamba'], check=True)
PY38 = Path('/content/g1-python38/bin/python')
env = os.environ.copy()
env['MAMBA_ROOT_PREFIX'] = '/content/mamba-root'
if not PY38.exists():
    subprocess.run([str(MAMBA_BIN), 'create','-y','-p','/content/g1-python38',
         '-c','conda-forge','python=3.8','pip'], check=True, env=env)
print('Python 3.8:', subprocess.check_output([str(PY38), '--version'], text=True).strip())

In [ ]:
from pathlib import Path
import subprocess, os

assert ARCHIVE.is_file(), f'Isaac Gym配布物がありません: {ARCHIVE}'
assert PY38.exists(), '先にPython 3.8環境を作成してください．'

EXTRACT = Path('/content/isaacgym_dist')
if not EXTRACT.exists():
    EXTRACT.mkdir(parents=True)
    subprocess.run(['tar','-xf',str(ARCHIVE),'-C',str(EXTRACT)], check=True)

isaac_python_dirs = [p for p in EXTRACT.rglob('python') if (p/'setup.py').is_file() and p.parent.name == 'isaacgym']
assert isaac_python_dirs, '配布物から isaacgym/python/setup.py が見つかりません．構成を確認してください．'
ISAAC_PY_DIR = isaac_python_dirs[0]
print('Isaac Gym Python package:', ISAAC_PY_DIR)

def pip(*args):
    subprocess.run([str(PY38),'-m','pip',*map(str,args)],check=True)

pip('install','--upgrade','pip','setuptools','wheel')
pip('install','numpy==1.23.5','gym==0.23.1','imageio','ninja','tensorboard')
pip('install','torch==2.3.1','torchvision==0.18.1','torchaudio==2.3.1',
    '--index-url','https://download.pytorch.org/whl/cu121')
# 既存GPU版PyTorchを維持し，旧Isaac Gymインストールの依存関係で置き換えない
pip('install','-e',str(ISAAC_PY_DIR),'--no-deps')
pip('install','-e',str(rsl))
pip('install','-e',str(repo))
print('依存ライブラリのインストールが終了しました．まだGPU動作の確認は済んでいません．')

In [ ]:
PY38 = Path('/content/g1-python38/bin/python')
assert PY38.is_file(), PY38
RUN_ENV = os.environ.copy()
RUN_ENV['LD_LIBRARY_PATH'] = '/content/g1-python38/lib:' + RUN_ENV.get('LD_LIBRARY_PATH','')
RUN_ENV['TORCH_EXTENSIONS_DIR'] = '/content/g1_torch_extensions'
RUN_ENV['PATH'] = str(PY38.parent) + os.pathsep + RUN_ENV.get('PATH','')
Path(RUN_ENV['TORCH_EXTENSIONS_DIR']).mkdir(exist_ok=True)
subprocess.run([str(PY38),'-m','pip','install','-q','ninja'],check=True,env=RUN_ENV)
p=subprocess.run([str(PY38),'-c',
 'from isaacgym import gymapi; import torch; from torch.utils.cpp_extension import is_ninja_available; '
 'print("CUDA:",torch.cuda.is_available(),"Ninja:",is_ninja_available())'],
 cwd=str(repo),env=RUN_ENV,text=True,capture_output=True)
print(p.stdout,p.stderr[-2000:]); assert p.returncode==0
assert 'CUDA: True Ninja: True' in p.stdout

## 3．P0・P1・P2の学習済み方策を確認

In [ ]:
MODELS={key:EXP/key/'checkpoints'/'model_6700.pt' for key in ('P0','P1','P2')}
for name,f in MODELS.items():
    assert f.is_file() and f.stat().st_size>100000, f
    print(name, f, round(f.stat().st_size/1024),'KiB')
print('学習済みファイルは読み取り専用で利用します．')

## 4．測定条件を設定

試行数や速度範囲を変えるときは**このセル**だけ変更し，それ以降のセルを再実行してください．ただし別条件の結果を残す場合は`RUN_NAME`も変えてください．

実験では方策ごとに同じ速度グリッドを，それぞれ別プロセスで実行します．

In [ ]:
import json, numpy as np
# まずは高速側の粗い探索．0.6と1.0は既知条件の再確認．
SPEEDS = [1.8, 1.9]
REPS = 8
DURATION_S = 18.0
SEED = 2026
RUN_NAME = 'fine_1p9_20261028'  # 再実験時は'fine'などに変更

assert len(SPEEDS) >= 2 and sorted(set(SPEEDS)) == SPEEDS
assert all(0 < x <= 4 for x in SPEEDS)
assert 1 <= REPS <= 16
assert DURATION_S <= 18.0, 'エピソード上限を越えないよう，最初は18秒以下にしてください'
CONFIG_FILE = Path('/content/g1_highspeed_experiment_config.json')
CONFIG_FILE.write_text(json.dumps({
    'speeds_mps': SPEEDS, 'reps': REPS, 'duration_s': DURATION_S,
    'seed': SEED, 'sample_every_steps': 5
}, indent=2),encoding='utf-8')
RUN_DIR = OUT_DIR / RUN_NAME
RUN_DIR.mkdir(parents=True,exist_ok=True)
print('指令速度:', SPEEDS)
print('並列G1の数:', REPS*len(SPEEDS))
print('シミュレーション:',len(SPEEDS),'速度 ×',REPS,'試行 ×',len(MODELS),'方策')
print('新規保存先:', RUN_DIR)

## 5．シミュレーション用スクリプトの生成（構文検査）

このセルで `/content/g1_highspeed_limit_eval.py` を生成します．生成したファイルはGoogle Driveにある公式実装を書き換えません．

In [ ]:
from pathlib import Path
import py_compile
SCRIPT=Path('/content/g1_highspeed_limit_eval.py')
SCRIPT.write_text("import argparse, csv, json, hashlib, sys\nfrom pathlib import Path\nimport numpy as np\nfrom isaacgym import gymapi, gymtorch  # must be before importing torch\nimport torch\nfrom legged_gym.envs import *\nfrom legged_gym.utils import task_registry, get_args\n\nap=argparse.ArgumentParser(add_help=False)\nap.add_argument('--checkpoint', required=True)\nap.add_argument('--label', required=True)\nap.add_argument('--out', required=True)\nap.add_argument('--experiment-config', required=True)\nopt, other=ap.parse_known_args()\nsys.argv=[sys.argv[0]]+other\nargs=get_args()\nassert args.task=='g1'\nex=json.loads(Path(opt.experiment_config).read_text(encoding='utf-8'))\nSPEEDS=tuple(float(v) for v in ex['speeds_mps'])\nREPS=int(ex['reps'])\nDURATION_S=float(ex['duration_s'])\nSEED=int(ex['seed'])\nSAMPLE_EVERY=int(ex.get('sample_every_steps',5))\nN=len(SPEEDS)*REPS\nargs.seed=SEED\nargs.num_envs=N\n\ncfg,train_cfg=task_registry.get_cfgs(name='g1')\ncfg.env.num_envs=N\ncfg.seed=SEED\ncfg.env.test=False\ncfg.noise.add_noise=False\ncfg.domain_rand.randomize_friction=False\ncfg.domain_rand.randomize_base_mass=False\ncfg.domain_rand.push_robots=False\ncfg.terrain.mesh_type='plane'\ncfg.terrain.curriculum=False\ncfg.commands.curriculum=False\ncfg.commands.heading_command=False\ncfg.commands.resampling_time=1_000_000.0\n\nenv,_=task_registry.make_env(name='g1',args=args,env_cfg=cfg)\nrunner,_=task_registry.make_alg_runner(env=env,name='g1',args=args,\n                                       train_cfg=train_cfg,log_root=None)\nckpt=torch.load(opt.checkpoint,map_location=env.device)\nassert 'model_state_dict' in ckpt\nrunner.alg.actor_critic.load_state_dict(ckpt['model_state_dict'])\npolicy=runner.get_inference_policy(device=env.device)\nassert runner.alg.actor_critic.memory_a.hidden_states is None\n\ndt=float(env.dt)\nSTEPS=int(round(DURATION_S/dt))\nassert abs(STEPS*dt-DURATION_S)<1e-3\nassert STEPS<env.max_episode_length, 'episode timeout would confound fall check'\nassert env.num_envs==N\nprint('Evaluation settings:',opt.label,'envs',N,'steps',STEPS,'dt',dt,flush=True)\n\n# Each policy uses precisely the same random initial states.\ntorch.manual_seed(SEED)\ntorch.cuda.manual_seed_all(SEED)\nperturb=(torch.rand((N,env.num_dof),device=env.device)-0.5)*0.04\nenv.dof_pos[:]=env.default_dof_pos+perturb\nenv.dof_vel[:]=0.0\nenv.root_states[:,7:13]=0.0\nenv.gym.set_dof_state_tensor(env.sim,gymtorch.unwrap_tensor(env.dof_state))\nenv.gym.set_actor_root_state_tensor(env.sim,gymtorch.unwrap_tensor(env.root_states))\nenv.base_lin_vel[:]=0.0\nenv.base_ang_vel[:]=0.0\nenv.episode_length_buf[:]=0\nenv.actions[:]=0.0\nenv.last_actions[:]=0.0\n\ntargets=torch.tensor([v for v in SPEEDS for _ in range(REPS)],device=env.device,dtype=torch.float32)\nenv.commands[:]=0.0\nenv.commands[:,0]=targets\nif hasattr(env,'phase'):env.phase[:]=0.0\nif hasattr(env,'phase_left'):env.phase_left[:]=0.0\nif hasattr(env,'phase_right'):env.phase_right[:]=0.5\nif hasattr(env,'leg_phase'):\n    env.leg_phase[:,0]=0.0\n    env.leg_phase[:,1]=0.5\nenv.compute_observations()\nobs=env.get_observations()\nassert obs.shape==(N,47)\nassert torch.allclose(env.commands[:,0],targets)\n\nroot=env.root_states.detach().cpu().numpy().copy()\nq=env.dof_pos.detach().cpu().numpy().copy()\nqd=env.dof_vel.detach().cpu().numpy().copy()\ninitial_sha=hashlib.sha256(np.ascontiguousarray(np.concatenate([\n    root[:,:13].flatten(),q.flatten(),qd.flatten()]).astype('<f4')).tobytes()).hexdigest()\nprint('initial sha256:',initial_sha,flush=True)\n\nalive=torch.ones(N,dtype=torch.bool,device=env.device)\nfalls=torch.full((N,),float('nan'),device=env.device)\nmin_height=np.full(N,np.inf)\nmax_abs_roll=np.zeros(N)\nmax_abs_pitch=np.zeros(N)\nsteady_n=np.zeros(N,dtype=np.int64)\nsteady_body_sum=np.zeros(N)\nsteady_world_sum=np.zeros(N)\nsteady_body_sqerr=np.zeros(N)\nlast_body=np.full(N,np.nan)\nlast_world=np.full(N,np.nan)\nraw=[]\n\nwith torch.inference_mode():\n  for step in range(STEPS):\n    t=round((step+1)*dt,5)\n    was_alive=alive.clone()\n    actions=policy(obs.detach())\n    obs,_,_,dones,_=env.step(actions.detach())\n    dones=dones.bool()\n    just_ended=was_alive & dones\n    falls[just_ended]=t\n    alive &= ~dones\n    runner.alg.actor_critic.memory_a.reset(dones)\n    good=(was_alive & ~dones).detach().cpu().numpy()\n\n    body_vx=env.base_lin_vel[:,0].detach().cpu().numpy()\n    world_vx=env.root_states[:,7].detach().cpu().numpy()\n    world_x=env.root_states[:,0].detach().cpu().numpy()\n    rpy=env.rpy[:,:3].detach().cpu().numpy()\n    height=env.base_pos[:,2].detach().cpu().numpy()\n\n    min_height[good]=np.minimum(min_height[good],height[good])\n    max_abs_roll[good]=np.maximum(max_abs_roll[good],np.abs(rpy[good,0]))\n    max_abs_pitch[good]=np.maximum(max_abs_pitch[good],np.abs(rpy[good,1]))\n    last_body[good]=body_vx[good]\n    last_world[good]=world_vx[good]\n    if t>=4.0:\n       idx=np.flatnonzero(good)\n       steady_n[idx]+=1\n       steady_body_sum[idx]+=body_vx[idx]\n       steady_world_sum[idx]+=world_vx[idx]\n       cmd=np.asarray(SPEEDS,dtype=float)[idx//REPS]\n       steady_body_sqerr[idx]+=(body_vx[idx]-cmd)**2\n\n    if (step+1)%SAMPLE_EVERY==0 or step==0:\n       for j in np.flatnonzero(good):\n          raw.append({'model':opt.label,'command_mps':SPEEDS[j//REPS],\n               'rep':j%REPS,'time_s':t,'body_vx_mps':float(body_vx[j]),\n               'world_vx_mps':float(world_vx[j]),'world_x_m':float(world_x[j]),\n               'roll_rad':float(rpy[j,0]),'pitch_rad':float(rpy[j,1]),\n               'yaw_rad':float(rpy[j,2]),'base_height_m':float(height[j])})\n    if not bool(alive.any().item()):\n       print('All environments terminated by step',step+1,flush=True)\n       break\n\nfall_np=falls.detach().cpu().numpy()\nrows=[]\nfor j in range(N):\n   v=SPEEDS[j//REPS]\n   survived=not np.isfinite(fall_np[j])\n   denom=max(1,steady_n[j])\n   mean_body=steady_body_sum[j]/denom if steady_n[j]>0 else float('nan')\n   mean_world=steady_world_sum[j]/denom if steady_n[j]>0 else float('nan')\n   rmse=np.sqrt(steady_body_sqerr[j]/denom) if steady_n[j]>0 else float('nan')\n   tolerance=max(.10,.15*v)\n   tracking=bool(survived and np.isfinite(mean_world) and np.isfinite(mean_body)\n                 and abs(mean_world-v)<=tolerance and abs(mean_body-v)<=tolerance)\n   rows.append({'model':opt.label,'command_mps':v,'rep':j%REPS,\n      'survived_18s':int(survived),\n      'termination_time_s':float(fall_np[j]) if not survived else '',\n      'steady_body_vx_mps':mean_body,'steady_world_vx_mps':mean_world,\n      'steady_body_rmse_mps':rmse,'speed_tolerance_mps':tolerance,\n      'survived_and_tracked':int(tracking),\n      'min_base_height_m':float(min_height[j]) if np.isfinite(min_height[j]) else '',\n      'max_abs_roll_rad':float(max_abs_roll[j]),\n      'max_abs_pitch_rad':float(max_abs_pitch[j]),\n      'observed_steady_samples':int(steady_n[j])})\n\nout=Path(opt.out)\nout.mkdir(parents=True,exist_ok=True)\ndef save_csv(filename,rs):\n   assert rs, filename\n   with (out/filename).open('w',newline='',encoding='utf-8') as f:\n      w=csv.DictWriter(f,fieldnames=list(rs[0].keys()))\n      w.writeheader();w.writerows(rs)\nsave_csv('trials_'+opt.label+'.csv',rows)\nsave_csv('sampled_trajectories_'+opt.label+'.csv',raw)\n(out/('config_'+opt.label+'.json')).write_text(json.dumps({\n   **ex,'label':opt.label,'checkpoint':str(opt.checkpoint),\n   'actual_dt_s':dt,'num_envs':N,'initial_state_sha256':initial_sha,\n   'root_and_joint_velocity_initial':'0',\n   'initial_joint_angles':'default + uniform(-0.02,0.02) rad',\n   'command_clamping':'no clipping applied; direct manual env.commands assignment',\n   'tracking_tolerance':'max(0.10 m/s, 15% of command)',\n   'steady_window_s':'[4, 18] for surviving trials',\n   'episode_done_interpreted_as':'termination/fall proxy'},\n   indent=2,ensure_ascii=False),encoding='utf-8')\nprint('saved',opt.label,'rows',len(rows),'survived',sum(x['survived_18s'] for x in rows),\n      'track',sum(x['survived_and_tracked'] for x in rows),'of',N,flush=True)\n",encoding='utf-8')
py_compile.compile(str(SCRIPT),doraise=True)
print('構文OK:',SCRIPT)

## 6．P0・P1・P2の高速歩行を順に実行（T4 GPU）

各方策を独立したPythonプロセスで実行します．プログラムが学習済みモデルを変更することはありません．各モデルで同じ初期身体状態を使ったことは次のセルでSHA256により確認します．

In [ ]:
import time, shutil, subprocess
LOCAL_DIR=Path('/content/g1_highspeed_checkpoints')
LOCAL_DIR.mkdir(parents=True,exist_ok=True)
for label,src in MODELS.items():
    dest=LOCAL_DIR / (label+'.pt')
    shutil.copy2(src,dest)
    log=RUN_DIR/f'eval_{label}.log'
    cmd=[str(PY38),str(SCRIPT),'--task=g1','--headless',
         '--sim_device=cuda:0','--rl_device=cuda:0',
         '--num_envs='+str(len(SPEEDS)*REPS),'--seed='+str(SEED),
         '--checkpoint',str(dest),'--label',label,
         '--experiment-config',str(CONFIG_FILE),'--out',str(RUN_DIR)]
    print('評価開始:',label,flush=True)
    start=time.time()
    with log.open('w',encoding='utf-8') as f:
        proc=subprocess.run(cmd,cwd=str(repo),env=RUN_ENV,stdout=f,stderr=subprocess.STDOUT)
    tail='\n'.join(log.read_text(encoding='utf-8',errors='replace').splitlines()[-18:])
    print(label,'終了コード:',proc.returncode,'経過分:',round((time.time()-start)/60,2),'\n'+tail,flush=True)
    assert proc.returncode==0,f'{label}で失敗しました．ログを確認してください: {log}'
print('P0，P1，P2の比較実験が終了しました．')

## 7．比較条件の同一性を確認し，非転倒率と実速度を集計

「非転倒である速度指令」と「実際にその指令速度に追従できる速度」を分けます．「追従した」と数える条件は，18秒間生存したうえで，**定常区間の身体前方速度とワールド前方速度がともに指令値±max(0.10 m/s,15%)**に入っていることです．

In [ ]:
import pandas as pd, numpy as np, json
from IPython.display import display
import matplotlib.pyplot as plt
TRIALS=[]
SHA=[]
for m in MODELS:
    f=RUN_DIR/f'trials_{m}.csv'
    assert f.is_file(),f
    df=pd.read_csv(f)
    assert len(df)==len(SPEEDS)*REPS
    TRIALS.append(df)
    p=json.loads((RUN_DIR/f'config_{m}.json').read_text(encoding='utf-8'))
    SHA.append(p['initial_state_sha256'])
assert len(set(SHA))==1, f'P0/P1/P2で初期条件が一致していません: {SHA}'
trials=pd.concat(TRIALS,ignore_index=True)
assert trials.groupby(['model','command_mps']).size().eq(REPS).all()
assert trials.survived_and_tracked.le(trials.survived_18s).all()
print('3方策の初期条件SHA256が一致:', SHA[0])

summary=trials.groupby(['model','command_mps'],as_index=False).agg(
    n=('rep','size'),survived=('survived_18s','sum'),
    tracked=('survived_and_tracked','sum'))
summary['survival_rate']=summary.survived/summary.n
summary['tracking_survival_rate']=summary.tracked/summary.n

# 平均実速度は，少なくとも18秒生存した試行だけから求める．
survivors=trials.query('survived_18s == 1')
world_means=survivors.groupby(['model','command_mps'],as_index=False).agg(
    survivor_actual_world_vx_mean_mps=('steady_world_vx_mps','mean'),
    survivor_actual_world_vx_sd_mps=('steady_world_vx_mps','std'),
    survivor_actual_body_vx_mean_mps=('steady_body_vx_mps','mean'))
summary=summary.merge(world_means,on=['model','command_mps'],how='left')
summary.to_csv(RUN_DIR/'high_speed_summary.csv',index=False)
trials.to_csv(RUN_DIR/'high_speed_all_trials.csv',index=False)
display(summary.round(3))

# 「試験速度の中で最も高い，8/8非転倒かつ追従達成」と「最も高い，8/8非転倒」をそれぞれ列挙．
limits=[]
for model,df in summary.groupby('model'):
    all_survived=df[df.survived.eq(REPS)]
    all_tracked=df[df.tracked.eq(REPS)]
    limits.append({
        'model':model,
        'highest_tested_command_all_survived_mps':all_survived.command_mps.max() if len(all_survived) else np.nan,
        'highest_tested_command_all_survived_and_tracked_mps':all_tracked.command_mps.max() if len(all_tracked) else np.nan,
        'highest_tested_command_mps':max(SPEEDS),
        'highest_tested_all_failed':int(df.loc[df.command_mps.eq(max(SPEEDS)),'survived'].iat[0]==0)})
limits=pd.DataFrame(limits)
limits.to_csv(RUN_DIR/'high_speed_limit_brief.csv',index=False)
display(limits)
#print('上端2.6m/sでも全試行が生存・追従する場合，限界はまだ測定できていません．')
print(f'上端{max(SPEEDS):.1f} m/sでも全試行が生存・追従する場合，限界はまだ測定できていません．')

## 8．図：速度指令に対する非転倒率と実測前進速度（CPU）

実線：P0，破線：P1，一点鎖線：P2．同じ色を通じて用います．

In [ ]:
STYLE={
    'P0':dict(color='#1f77b4',linestyle='-',marker='o'),
    'P1':dict(color='#ff7f0e',linestyle='--',marker='s'),
    'P2':dict(color='#d62728',linestyle='-.',marker='^')
}
fig,axs=plt.subplots(2,1,figsize=(11,9),sharex=True,layout='constrained')
for label in MODELS:
    d=summary[summary.model.eq(label)].sort_values('command_mps')
    st=STYLE[label]
    axs[0].plot(d.command_mps,100*d.survival_rate,label=label,linewidth=2.5,markersize=6,**st)
    axs[0].plot(d.command_mps,100*d.tracking_survival_rate,
                color=st['color'],linestyle=':',marker=st['marker'],
                alpha=.7,linewidth=1.7,markersize=4)
    axs[1].plot(d.command_mps,d.survivor_actual_world_vx_mean_mps,
                label=label,linewidth=2.5,markersize=6,**st)
axs[0].set(ylabel='Survival rate (%)',ylim=(-5,105),title='High-speed walking: survival (thick) and survival + tracking (dotted)')
axs[1].plot(SPEEDS,SPEEDS,'k:',alpha=.6,label='command = actual')
axs[1].set(xlabel='Commanded forward speed (m/s)',ylabel='Actual world-forward speed among survivors (m/s)')
for ax in axs:ax.grid(alpha=.3);ax.legend()
fig.savefig(RUN_DIR/'P012_survival_and_actual_speed.png',dpi=180)
plt.show()

fig,ax=plt.subplots(figsize=(11,5),layout='constrained')
for label in MODELS:
    d=summary[summary.model.eq(label)].sort_values('command_mps')
    ax.plot(d.command_mps,100*d.tracking_survival_rate,label=label,
            linewidth=2.5,markersize=6,**STYLE[label])
ax.set(xlabel='Commanded speed (m/s)',ylabel='Survived 18s AND tracked speed (%)',
       ylim=(-5,105),title='Effective high-speed walking success rate')
ax.grid(alpha=.3);ax.legend()
fig.savefig(RUN_DIR/'P012_effective_success_rate.png',dpi=180)
plt.show()

## 9．転倒した試行の発生時刻・姿勢の確認（CPU）

転倒した試行があれば時刻と転倒前の記録を確認します．`done`にはエピソード終了の意味があるため，**終了理由をすべて機械的な転倒と断定しません**．速度の非追従は，転倒していなくても失敗として扱います．

In [ ]:
failed=trials.query('survived_18s == 0').copy()
if failed.empty:
    print('今回の範囲では18秒以内の終了は発生していません．速度上限が特定できたわけではありません．')
else:
    display(failed[['model','command_mps','rep','termination_time_s','min_base_height_m',
       'max_abs_roll_rad','max_abs_pitch_rad','observed_steady_samples']].sort_values(
       ['model','command_mps','rep']).round(3))
print('転倒はしないが速度追従しなかった試行数:',len(trials.query('survived_18s == 1 and survived_and_tracked == 0')))
print('出力先:',RUN_DIR)
print('すべての出力は既存の学習済みモデル・評価ファイルとは別フォルダーです．')

## 10．解釈と次の速度探索

- **全試行で非転倒**は「18秒・この初期条件・8試行で非転倒」の意味です．最大能力を数学的に証明するものではありません．
- 速度を上げてもG1が歩かず，指令に追従しない場合は「高速歩行に成功した」とは判定しません．
- 試験上限（2.6 m/s）まで全モデルが非転倒なら，その後さらに高速へ範囲を広げる必要があります．
- 速度範囲の中に成功から失敗への境界が現れたら，その近傍を0.05 m/s刻みで追加評価し，試行数・初期状態・外乱条件も増やしましょう．
- 対照的な結果が出ても，現時点のP0/P1/P2は1つの**学習シード**からの分岐学習です．再現性を確認するには別学習シードの反復が必要です．

In [ ]:
# 追加解析①：保存済みCSVのみを読む．Isaac Gym/GPUは使いません．
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# 先の高速歩行実験が保存したフォルダーを直接指定．
# Colabを再起動していても，このセルから単独実行できます．
RUN_DIR = OUT_DIR / RUN_NAME
#RUN_DIR = Path(
#    '/content/drive/MyDrive/Colab Notebooks/PhysicalAI応用編１/'
#    'G1_RL_Training/contact_tolerance_P0_P1_P2_from6500_20261024_075342/'
#    'evaluation/high_speed_limit_P012_coarse_20261024/coarse'
#)
if not RUN_DIR.is_dir():
    from google.colab import drive
    drive.mount('/content/drive')
assert RUN_DIR.is_dir(), f'結果フォルダーが見つかりません: {RUN_DIR}'

OUT_ANALYSIS = RUN_DIR / 'body_speed_yaw_analysis'
OUT_ANALYSIS.mkdir(exist_ok=True)
MODELS_ORDER = ('P0', 'P1', 'P2')
STYLES = {
    'P0': dict(color='#1f77b4', linestyle='-',  marker='o'),
    'P1': dict(color='#ff7f0e', linestyle='--', marker='s'),
    'P2': dict(color='#d62728', linestyle='-.', marker='^'),
}

trials_body = pd.read_csv(RUN_DIR / 'high_speed_all_trials.csv')
trajectories = pd.concat([
    pd.read_csv(RUN_DIR / f'sampled_trajectories_{m}.csv')
    for m in MODELS_ORDER
], ignore_index=True)

required_trial = {'model', 'command_mps', 'rep', 'survived_18s',
                  'steady_body_vx_mps', 'steady_world_vx_mps',
                  'speed_tolerance_mps', 'survived_and_tracked'}
required_trace = {'model', 'command_mps', 'rep', 'time_s',
                  'yaw_rad', 'body_vx_mps', 'world_vx_mps', 'world_x_m'}
assert required_trial.issubset(trials_body), required_trial - set(trials_body)
assert required_trace.issubset(trajectories), required_trace - set(trajectories)
assert set(trials_body.model) == set(MODELS_ORDER)
assert not trials_body.duplicated(['model','command_mps','rep']).any()

# 既存の判定は「身体前進速度と世界X速度の両方が許容幅内」だった．
# 今回は身体前進速度の追従だけを独立して判定する．
trials_body['body_tracking_18s'] = (
    trials_body['survived_18s'].eq(1)
    & trials_body['steady_body_vx_mps'].notna()
    & (trials_body['steady_body_vx_mps'] - trials_body['command_mps']).abs().le(
        trials_body['speed_tolerance_mps'] + 1e-10)
).astype(int)
trials_body['world_tracking_18s'] = (
    trials_body['survived_18s'].eq(1)
    & trials_body['steady_world_vx_mps'].notna()
    & (trials_body['steady_world_vx_mps'] - trials_body['command_mps']).abs().le(
        trials_body['speed_tolerance_mps'] + 1e-10)
).astype(int)
assert (
    (trials_body['body_tracking_18s'] & trials_body['world_tracking_18s'])
    == trials_body['survived_and_tracked']
).all(), '以前の追従判定を再現できません．データを確認してください．'

# yaw角は±πで折り返されるので，試行ごとにunwrapしてから初期角を引く．
trajectories = trajectories.sort_values(
    ['model','command_mps','rep','time_s']
).reset_index(drop=True)
GROUP_KEYS = ['model', 'command_mps', 'rep']
trajectories['yaw_unwrapped_rad'] = trajectories.groupby(GROUP_KEYS)['yaw_rad'].transform(
    lambda x: np.unwrap(x.to_numpy())
)
trajectories['yaw_change_deg'] = np.rad2deg(
    trajectories['yaw_unwrapped_rad']
    - trajectories.groupby(GROUP_KEYS)['yaw_unwrapped_rad'].transform('first')
)
trajectories['world_x_change_m'] = (
    trajectories['world_x_m']
    - trajectories.groupby(GROUP_KEYS)['world_x_m'].transform('first')
)

print('CSVの読み込み・既存成功判定との整合性確認：OK')
print('試行:', len(trials_body), '軌跡記録:', len(trajectories))
print('入力:', RUN_DIR)
print('新しい解析結果:', OUT_ANALYSIS)


In [ ]:
# 追加解析②：指令速度と実測身体前進速度，非転倒率を分離して比較
alive = trials_body[trials_body['survived_18s'].eq(1)].copy()
counts = trials_body.groupby(['model', 'command_mps'], as_index=False).agg(
    n=('rep', 'size'),
    n_survived=('survived_18s', 'sum'),
    n_body_tracking=('body_tracking_18s', 'sum'),
    n_world_tracking=('world_tracking_18s', 'sum'),
    n_both_tracking=('survived_and_tracked', 'sum'),
)
means = alive.groupby(['model', 'command_mps'], as_index=False).agg(
    body_vx_mean=('steady_body_vx_mps', 'mean'),
    body_vx_sd=('steady_body_vx_mps', 'std'),
    world_vx_mean=('steady_world_vx_mps', 'mean'),
    world_vx_sd=('steady_world_vx_mps', 'std'),
)
comparison = counts.merge(means, on=['model','command_mps'], how='left')
comparison['survival_pct'] = 100 * comparison['n_survived'] / comparison['n']
comparison['body_tracking_pct'] = 100 * comparison['n_body_tracking'] / comparison['n']
comparison['world_tracking_pct'] = 100 * comparison['n_world_tracking'] / comparison['n']
comparison['both_tracking_pct'] = 100 * comparison['n_both_tracking'] / comparison['n']
comparison.to_csv(OUT_ANALYSIS / 'P012_body_speed_vs_world_speed_summary.csv', index=False)
trials_body.to_csv(OUT_ANALYSIS / 'P012_body_tracking_by_trial.csv', index=False)

display(comparison[[
    'model','command_mps','n_survived','n_body_tracking',
    'n_both_tracking','body_vx_mean','world_vx_mean'
]].round(3))

fig, axes = plt.subplots(3, 1, figsize=(11, 12), sharex=True, layout='constrained')
speeds = sorted(comparison.command_mps.unique())
for model in MODELS_ORDER:
    d = comparison.query('model == @model').sort_values('command_mps')
    st = STYLES[model]
    # 生存者がいない速度は平均速度を描画しない．平均±試行間SD．
    axes[0].errorbar(d.command_mps, d.body_vx_mean,
                     yerr=d.body_vx_sd.fillna(0), fmt=st['marker'],
                     color=st['color'], linestyle=st['linestyle'],
                     capsize=3, linewidth=2, markersize=6, label=model)
    axes[1].plot(d.command_mps, d.world_vx_mean,
                 color=st['color'], linestyle=st['linestyle'],
                 marker=st['marker'], linewidth=2, label=model)
    axes[2].plot(d.command_mps, d.survival_pct,
                 color=st['color'], linestyle=st['linestyle'],
                 marker=st['marker'], linewidth=2.5, label=f'{model}: survived')
    axes[2].plot(d.command_mps, d.body_tracking_pct,
                 color=st['color'], linestyle=':', marker=st['marker'],
                 linewidth=1.6, alpha=.75, label=f'{model}: survived + body-speed tracked')
for ax in axes[:2]:
    ax.plot(speeds, speeds, 'k:', alpha=.5, label='commanded speed')
axes[0].set(ylabel='Body-forward velocity (m/s)',
            title='Body-forward speed of 18-s survivors (mean +/- SD)')
axes[1].set(ylabel='World-X velocity (m/s)',
            title='World-forward component of 18-s survivors')
axes[2].set(xlabel='Commanded body-forward speed (m/s)',
            ylabel='Percentage of 8 trials (%)',
            ylim=(-5, 105), title='Survival vs body-forward-speed tracking')
for ax in axes:
    ax.grid(alpha=.25)
    ax.legend(fontsize=9, ncol=2)
fig.savefig(OUT_ANALYSIS / 'P012_body_world_speed_and_survival.png', dpi=160)
plt.show()

print('注意：従来の「追従成功」は身体速度と世界X速度の両方を要求．')
print('今回の点線は「非転倒かつ身体速度だけを許容範囲内」として再計算．')



## 11．追加解析：身体前進速度・非転倒率・進行方向（CPUのみ）

**重要な訂正：** このノートブックの従来の `survived_and_tracked` は，`steady_body_vx_mps` **と** `steady_world_vx_mps` の両方が指令値の許容幅に入ることを要求していました．今回，新たに「身体前進速度のみ」の追従成功を別集計します．

以下の4つのコードセルは，**以前の1〜10節を再実行せず，Google Driveの保存CSVから単独実行できます**．初期化・Isaac Gym・GPUは不要です．新しい結果は `coarse/body_speed_yaw_analysis/` に保存し，従来のCSV・PNGを変更しません．

- 身体速度の時間平均は既存実験の4〜18秒区間．
- yawは角度の±180°の折り返しを取り除いて，試行開始時からの変化を計算．
- `episode done` は「シミュレーション終了／転倒代理指標」であり，すべて転倒と確定したわけではありません．

In [ ]:
# 追加解析③：yaw角の変化と世界X方向への変位を数値化
# 18秒間生存した試行だけについて最終角度を評価する．
last_sample = trajectories.groupby(GROUP_KEYS, sort=False).tail(1).copy()
endpoints = last_sample.merge(
    trials_body[GROUP_KEYS + ['survived_18s']], on=GROUP_KEYS, how='left', validate='one_to_one'
)
endpoints = endpoints[endpoints.survived_18s.eq(1)].copy()
# 終了直前のデータであることを確認（端末で実測した時刻を表示）．
endpoints['abs_final_yaw_deg'] = endpoints['yaw_change_deg'].abs()
heading = endpoints.groupby(['model','command_mps'],as_index=False).agg(
    n_survived=('rep','size'),
    median_abs_final_yaw_deg=('abs_final_yaw_deg','median'),
    min_abs_final_yaw_deg=('abs_final_yaw_deg','min'),
    max_abs_final_yaw_deg=('abs_final_yaw_deg','max'),
    median_world_x_displacement_m=('world_x_change_m','median'),
    median_last_time_s=('time_s','median'),
)
heading.to_csv(OUT_ANALYSIS / 'P012_heading_endpoints_by_speed.csv', index=False)
endpoints.to_csv(OUT_ANALYSIS / 'P012_heading_endpoints_by_trial.csv', index=False)

display(heading[heading.command_mps.ge(1.4)].round(2))

fig, axs = plt.subplots(2, 1, figsize=(11, 9), sharex=True, layout='constrained')
for model in MODELS_ORDER:
    d = heading.query('model == @model').sort_values('command_mps')
    st = STYLES[model]
    axs[0].plot(d.command_mps, d.median_abs_final_yaw_deg,
                label=model, linewidth=2, markersize=6, **st)
    axs[1].plot(d.command_mps, d.median_world_x_displacement_m,
                label=model, linewidth=2, markersize=6, **st)
axs[0].set(ylabel='|Final yaw change| (degrees)',
           title='Absolute change of heading at 18 s (median of survivors)')
axs[1].set(xlabel='Commanded body-forward speed (m/s)',
           ylabel='World-X displacement (m)',
           title='Progress along initial world-X direction (median of survivors)')
for ax in axs:
    ax.grid(alpha=.25)
    ax.legend()
fig.savefig(OUT_ANALYSIS / 'P012_final_yaw_and_world_x_displacement.png', dpi=160)
plt.show()

print('yaw角の変化は各試行の最初の保存サンプル（約0.02秒）を基準にしています．')
print('世界X方向の変位は軌跡長ではありません．世界Y方向位置は今回のCSVにありません．')


In [ ]:
# 追加解析④：yaw角時系列．細線は各試行，太線は生存試行の中央値
# 点線の細線は18秒以内に終了した試行です．
SHOW_SPEEDS = [1.8, 1.9]
for command_speed in SHOW_SPEEDS:
    fig, ax = plt.subplots(figsize=(11, 5), layout='constrained')
    for model in MODELS_ORDER:
        st = STYLES[model]
        g_model = trajectories[
            trajectories.model.eq(model)
            & np.isclose(trajectories.command_mps, command_speed)
        ]
        valid_reps = trials_body[
            trials_body.model.eq(model)
            & np.isclose(trials_body.command_mps, command_speed)
            & trials_body.survived_18s.eq(1)
        ].rep.tolist()
        for rep, d in g_model.groupby('rep'):
            survived = (rep in valid_reps)
            ax.plot(d.time_s, d.yaw_change_deg,
                    color=st['color'], linestyle='-' if survived else ':',
                    alpha=.17 if survived else .32, linewidth=1)
        # 中央値は同じ時刻にデータがある「18秒生存した試行」のみから算出
        good = g_model[g_model.rep.isin(valid_reps)]
        if not good.empty:
            med = good.groupby('time_s',as_index=False)['yaw_change_deg'].median()
            ax.plot(med.time_s, med.yaw_change_deg,
                    color=st['color'], linestyle=st['linestyle'],
                    linewidth=2.6, label=f'{model}: {len(valid_reps)}/8 survived')
        else:
            ax.plot([], [], color=st['color'], linestyle=st['linestyle'],
                    linewidth=2.6, label=f'{model}: 0/8 survived')
    ax.axhline(0, color='black', linewidth=.8, alpha=.5)
    ax.set(xlim=(0, 18), xlabel='Time (s)',
           ylabel='Heading change (degrees, unwrapped)',
           title=f'Heading change after immediate start / command {command_speed:.1f} m/s')
    ax.grid(alpha=.25)
    ax.legend()
    fig.savefig(OUT_ANALYSIS / f'P012_yaw_time_{command_speed:.1f}.png', dpi=160)
    plt.show()

print('図中の太線＝非転倒試行の中央値．細線＝各試行（点線は18秒未満で終了）．')
print('方策間で生存試行数が違うため，中央値の比較には生存者選択の影響があります．')
